### Ingest constructor.json file
1. Read the file using spark dataframe reader API
2. Add metadata columns
   - Source File
    - ingestion timestamp
3. Write to bronze delta table       

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

In [0]:
source_file = f"{landing_folder_path}/constructors.json"
table_name = f"{catalog_name}.{bronze_schema}.constructors"

In [0]:
constructor_schema = 'constructorId string, name string, nationality string, url string'

#### Step-1 Read the file using spark dataframe reader API


In [0]:
constructor_df = (
    spark.read.format('json')
    .option('header','true')
    .option('mode','FAILFAST')
    .schema(constructor_schema)
    .option('multiline','true')
    .load(source_file)
)

In [0]:
display(constructor_df)

#### Step-2. Add metadata columns
   - Source File
   - ingestion timestamp

In [0]:
constructor_final_df = add_ingestion_metadata(constructor_df)

In [0]:
(
    constructor_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.constructors